# 🎬 VP Library — Embed ภาพลง Supabase

Notebook นี้ใช้ embed ภาพทุกฉากใน VP Library ด้วย **mCLIP** แล้วบันทึก vector ลง Supabase

### วิธีใช้
1. ใส่ `SUPABASE_SERVICE_KEY` ใน Cell 2
2. กด **Runtime → Run all** แล้วรอ
3. เสร็จแล้วปิดได้เลย — ไม่ต้องรันใหม่ถ้าไม่มีภาพใหม่

> ⏱ ใช้เวลาประมาณ **5–10 นาที** สำหรับ 25 ฉาก (โหลด model ครั้งแรก ~2 นาที)

In [ ]:
# ── Cell 1: ติดตั้ง package ──────────────────────────────────────
!pip install sentence-transformers supabase pillow requests -q
print('✅ ติดตั้งเสร็จ')

In [ ]:
# ── Cell 2: ตั้งค่า — ใส่ Service Key ตรงนี้ ──────────────────────
# หา key ได้ที่: Supabase Dashboard → Project Settings → API → service_role

SUPABASE_URL         = 'https://pgaqdqbjyewwckpslyvx.supabase.co'
SUPABASE_SERVICE_KEY = 'YOUR_SERVICE_ROLE_KEY_HERE'   # ← ใส่ key ตรงนี้

# ถ้าอยากข้าม scene ที่ embed แล้ว ตั้งเป็น True (แนะนำ)
SKIP_ALREADY_INDEXED = True

print('✅ Config พร้อม')
print(f'   URL: {SUPABASE_URL}')
print(f'   Key: {SUPABASE_SERVICE_KEY[:20]}...' if len(SUPABASE_SERVICE_KEY) > 20 else '   ⚠️  ยังไม่ได้ใส่ key!')

In [ ]:
# ── Cell 3: โหลด mCLIP model + เชื่อมต่อ Supabase ─────────────────
from sentence_transformers import SentenceTransformer
from supabase import create_client
import requests
from PIL import Image
from io import BytesIO
from datetime import datetime, timezone

print('🔄 กำลังโหลด mCLIP model (~400MB)...')
model = SentenceTransformer('sentence-transformers/clip-ViT-B-32-multilingual-v1')
print('✅ โหลด model เสร็จ')

print('🔄 เชื่อมต่อ Supabase...')
sb = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)
print('✅ เชื่อมต่อ Supabase เสร็จ')

In [ ]:
# ── Cell 4: ดึงรายการฉากจาก Supabase ──────────────────────────────
result = sb.from_('scenes').select('id, image_url, thumb_url, mclip_indexed_at').order('sort_order').execute()
all_scenes = result.data or []

if SKIP_ALREADY_INDEXED:
    to_embed = [s for s in all_scenes if not s.get('mclip_indexed_at') and s.get('image_url')]
else:
    to_embed = [s for s in all_scenes if s.get('image_url')]

print(f'📊 ฉากทั้งหมด: {len(all_scenes)}')
print(f'✅ embed แล้ว: {len(all_scenes) - len(to_embed)}')
print(f'⏳ รอ embed:  {len(to_embed)}')

if not to_embed:
    print('\n🎉 ทุกฉาก embed แล้ว! ไม่ต้องทำอะไรเพิ่ม')
else:
    print('\nฉากที่จะ embed:')
    for s in to_embed:
        print(f'  • {s["id"]} — {s["image_url"][:60]}...')

In [ ]:
# ── Cell 5: Embed ทุกฉาก → บันทึกลง Supabase ───────────────────────
if not to_embed:
    print('🎉 ไม่มีฉากที่ต้อง embed')
else:
    success, failed = 0, 0
    errors = []

    for i, scene in enumerate(to_embed):
        scene_id  = scene['id']
        image_url = scene['image_url']
        print(f'[{i+1}/{len(to_embed)}] 🔄 {scene_id}...', end=' ')

        try:
            # 1. ดาวน์โหลดภาพ
            img_res = requests.get(image_url, timeout=30)
            img_res.raise_for_status()
            img = Image.open(BytesIO(img_res.content)).convert('RGB')

            # 2. Encode ด้วย mCLIP
            vector = model.encode(img).tolist()

            # 3. บันทึกลง Supabase
            now = datetime.now(timezone.utc).isoformat()
            sb.from_('scenes').update({
                'mclip_vector':     vector,
                'mclip_indexed_at': now,
            }).eq('id', scene_id).execute()

            success += 1
            print(f'✅ dim={len(vector)}')

        except Exception as e:
            failed += 1
            errors.append((scene_id, str(e)))
            print(f'❌ {e}')

    print(f'\n══════════════════════════════')
    print(f'✅ สำเร็จ:   {success} ฉาก')
    print(f'❌ ล้มเหลว:  {failed} ฉาก')
    if errors:
        print('\nรายละเอียด error:')
        for sid, err in errors:
            print(f'  {sid}: {err}')
    print('\n🎉 เสร็จสิ้น! ตอนนี้ vp-search.html ค้นหาภาพได้แล้ว')

In [ ]:
# ── Cell 6 (Optional): ตรวจสอบผลลัพธ์ ─────────────────────────────
result2 = sb.from_('scenes').select('id, mclip_indexed_at').order('sort_order').execute()
rows = result2.data or []

indexed   = [r for r in rows if r.get('mclip_indexed_at')]
unindexed = [r for r in rows if not r.get('mclip_indexed_at')]

print(f'📊 สรุปสถานะ Supabase:')
print(f'  ✅ มี vector แล้ว: {len(indexed)} ฉาก')
print(f'  ⏳ ยังไม่มี vector: {len(unindexed)} ฉาก')

if unindexed:
    print('\nฉากที่ยังไม่มี vector:')
    for r in unindexed:
        print(f'  • {r["id"]}')
else:
    print('\n🎉 ทุกฉากมี vector ครบแล้ว!')